# COMP5318 Assignment 1: Rice Classification

##### Group number: 69
##### Student 1 SID: 540207680
##### Student 2 SID: 530181464

## Table of Contents

- [1. Data Pre-processing](#1-data-pre-processing)
- [2. Build Classifiers](#2-build-classifiers)
  - [Part 1: Cross-validation without parameter tuning](#part-1-cross-validation-without-parameter-tuning)
  - [Part 2: Cross-validation with parameter tuning](#part-2-cross-validation-with-parameter-tuning)
  - [Part 2: Results](#part-2-results)
- [3. Reflection and Discussion](#3-reflection-and-discussion)


## **1. Data Pre-processing**

In [ ]:
# Import all libraries
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Load the rice dataset from the assignment folder
data_path = Path("rice-final2.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

riceDataFrame = pd.read_csv(data_path, na_values="?")

if riceDataFrame.shape[1] < 2:
    raise ValueError("The dataset must contain at least one feature column and one class column.")

X = riceDataFrame.iloc[:, :-1].copy()
y = riceDataFrame.iloc[:, -1].copy()

print("Dataset shape:", riceDataFrame.shape)
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Class values:", sorted(y.astype(str).str.strip().unique()))


In [ ]:
# Pre-process the dataset in a reusable way so it works with other files of the same format.
def preprocess_dataset(X, y):
    """Fill missing values, normalize features, and encode class labels."""
    if X.empty:
        raise ValueError("Feature matrix is empty.")
    if y.empty:
        raise ValueError("Target vector is empty.")

    # Convert feature columns to numeric so non-numeric placeholders become NaN.
    X_numeric = X.apply(pd.to_numeric, errors="coerce")

    # Guard against columns that contain no usable values at all.
    fully_missing_cols = X_numeric.columns[X_numeric.isna().all()].tolist()
    if fully_missing_cols:
        raise ValueError(f"These feature columns contain only missing or invalid values: {fully_missing_cols}")

    # Replace missing values with the mean of each column.
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(X_numeric)

    # Scale each feature to the [0, 1] range.
    scaler = MinMaxScaler()
    X_processed = scaler.fit_transform(X_imputed)

    # Encode class labels class1 -> 0 and class2 -> 1.
    y_clean = y.astype(str).str.strip()
    label_map = {"class1": 0, "class2": 1}
    y_processed = y_clean.map(label_map)

    if y_processed.isnull().any():
        unknown_labels = sorted(y_clean[y_processed.isnull()].unique())
        raise ValueError(f"Unexpected class labels found: {unknown_labels}")

    return X_processed, y_processed.astype(int).to_numpy(), imputer, scaler

XProcessed, yProcessed, imputer, scaler = preprocess_dataset(X, y)


In [ ]:
# Print the first ten rows of the pre-processed dataset using four decimal places.
def print_data(X, y, n_rows=10):
    """Print the first n_rows of a numpy feature matrix and label vector."""
    if len(X) != len(y):
        raise ValueError("Feature matrix and target vector must have the same number of rows.")

    rows_to_print = min(n_rows, len(X))
    for example_num in range(rows_to_print):
        feature_text = ",".join(f"{feature:.4f}" for feature in X[example_num])
        print(f"{feature_text},{int(y[example_num])}")

print_data(XProcessed, yProcessed)


In [ ]:
# Display the first ten rows as a formatted table for a cleaner notebook preview.
from IPython.display import display

def preview_data_table(X, y, n_rows=10):
    """Display the first n_rows as a styled table without changing the preprocessing logic."""
    if len(X) != len(y):
        raise ValueError("Feature matrix and target vector must have the same number of rows.")

    rows_to_show = min(n_rows, len(X))
    feature_cols = [f"Feature {i + 1}" for i in range(X.shape[1])]
    preview_df = pd.DataFrame(X[:rows_to_show], columns=feature_cols)
    preview_df["Class"] = y[:rows_to_show].astype(int)

    styled = (
        preview_df.style
        .format({col: "{:.4f}" for col in feature_cols})
        .set_caption("Pre-processed Dataset Preview")
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "16px"), ("font-weight", "600"), ("color", "#111827")]},
            {"selector": "th", "props": [("background-color", "#1f2937"), ("color", "white"), ("text-align", "center"), ("padding", "8px 10px")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "8px 10px")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("border", "1px solid #d1d5db"), ("border-radius", "8px"), ("overflow", "hidden")]},
        ])
    )
    display(styled)

preview_data_table(XProcessed, yProcessed)


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [ ]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [ ]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logRModel = LogisticRegression(
    max_iter=1000,
    random_state=0
)

logRScores = cross_val_score(
    estimator=logRModel,
    X=XProcessed,
    y=yProcessed,
    cv=cvKFold,
    scoring="accuracy"
)

logRAvgAccuracy = logRScores.mean()

# Fit a final Logistic Regression model for external test evaluation.
logRFinalModel = LogisticRegression(
    max_iter=1000,
    random_state=0
)
logRFinalModel.fit(XProcessed, yProcessed)


In [ ]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB

nbModel = GaussianNB()

nbScores = cross_val_score(
    estimator=nbModel,
    X=XProcessed,
    y=yProcessed,
    cv=cvKFold,
    scoring="accuracy"
)

nbAvgAccuracy = nbScores.mean()

# Fit a final Naive Bayes model for external test evaluation.
nbFinalModel = GaussianNB()
nbFinalModel.fit(XProcessed, yProcessed)


### Part 1 Results


In [ ]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {logRAvgAccuracy:.4f}")
print(f"NB average cross-validation accuracy: {nbAvgAccuracy:.4f}")

### Part 2: Cross-validation with parameter tuning

In [ ]:
# Split the data into training and test sets first so the same split can be reused by Decision Tree, AdaBoost, Gradient Boosting, Random Forest, and SVM.
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

xTrain, xTest, yTrain, yTest = train_test_split(
    XProcessed,
    yProcessed,
    test_size=0.2,
    stratify=yProcessed,
    random_state=0
)


In [ ]:
# KNN
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]

# p=1 -> Manhattan distance
# p=2 -> Euclidean distance
knnParameterGrid = {
    "n_neighbors": k,
    "p": p
}

knnModel = KNeighborsClassifier(metric="minkowski")

knnGridSearch = GridSearchCV(
    estimator=knnModel,
    param_grid=knnParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

knnGridSearch.fit(xTrain, yTrain)

knnBestModel = knnGridSearch.best_estimator_
knnBestK = knnGridSearch.best_params_["n_neighbors"]
knnBestP = knnGridSearch.best_params_["p"]
knnBestCvAccuracy = knnGridSearch.best_score_

knnTestPrediction = knnBestModel.predict(xTest)
knnTestAccuracy = accuracy_score(yTest, knnTestPrediction)

In [ ]:
# Decision Tree
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

decisionTreeParameterGrid = {
    "max_depth": max_depth,
    "min_samples_split": min_samples_split,
    "min_samples_leaf": min_samples_leaf
}

decisionTreeModel = DecisionTreeClassifier(random_state=0)

decisionTreeGridSearch = GridSearchCV(
    estimator=decisionTreeModel,
    param_grid=decisionTreeParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

decisionTreeGridSearch.fit(xTrain, yTrain)

decisionTreeBestModel = decisionTreeGridSearch.best_estimator_
decisionTreeBestMaxDepth = decisionTreeGridSearch.best_params_["max_depth"]
decisionTreeBestMinSamplesSplit = decisionTreeGridSearch.best_params_["min_samples_split"]
decisionTreeBestMinSamplesLeaf = decisionTreeGridSearch.best_params_["min_samples_leaf"]
decisionTreeBestCvAccuracy = decisionTreeGridSearch.best_score_

decisionTreeTestPrediction = decisionTreeBestModel.predict(xTest)
decisionTreeTestAccuracy = accuracy_score(yTest, decisionTreeTestPrediction)

In [ ]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score

adaBoostParameterGrid = {
    "n_estimators": n_estimators,
    "learning_rate": learning_rate
}

adaBoostModel = AdaBoostClassifier(random_state=0)

adaBoostGridSearch = GridSearchCV(
    estimator=adaBoostModel,
    param_grid=adaBoostParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

adaBoostGridSearch.fit(xTrain, yTrain)

adaBoostTestPrediction = adaBoostGridSearch.best_estimator_.predict(xTest)
adaBoostTestAccuracy = accuracy_score(yTest, adaBoostTestPrediction)

In [ ]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gradientBoostParameterGrid = {
    "max_depth": max_depth,
    "n_estimators": n_estimators,
    "learning_rate": learning_rate
}

gradientBoostModel = GradientBoostingClassifier(random_state=0)

gradientBoostGridSearch = GridSearchCV(
    estimator=gradientBoostModel,
    param_grid=gradientBoostParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

gradientBoostGridSearch.fit(xTrain, yTrain)

gradientBoostTestPrediction = gradientBoostGridSearch.best_estimator_.predict(xTest)
gradientBoostTestAccuracy = accuracy_score(yTest, gradientBoostTestPrediction)

In [ ]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to 'sqrt'.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

randomForestParameterGrid = {
    "n_estimators": n_estimators,
    "max_leaf_nodes": max_leaf_nodes
}

randomForestModel = RandomForestClassifier(
    criterion="entropy",      # information gain
    max_features="sqrt",
    random_state=0
)

randomForestGridSearch = GridSearchCV(
    estimator=randomForestModel,
    param_grid=randomForestParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

randomForestGridSearch.fit(xTrain, yTrain)

randomForestTestPrediction = randomForestGridSearch.best_estimator_.predict(xTest)
randomForestTestAccuracy = accuracy_score(yTest, randomForestTestPrediction)
randomForestMacroF1 = f1_score(yTest, randomForestTestPrediction, average="macro")
randomForestWeightedF1 = f1_score(yTest, randomForestTestPrediction, average="weighted")

In [ ]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]

# optional
kernel = ["rbf"]

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svmParameterGrid = {
    "C": C,
    "gamma": gamma,
    "kernel": kernel
}

svmModel = SVC(random_state=0)

svmGridSearch = GridSearchCV(
    estimator=svmModel,
    param_grid=svmParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

svmGridSearch.fit(xTrain, yTrain)

svmTestPrediction = svmGridSearch.best_estimator_.predict(xTest)
svmTestAccuracy = accuracy_score(yTest, svmTestPrediction)


### Part 2: Results

In [ ]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
# Print results for each classifier here.
# All the reported results should be printed to 4 decimal places except for integers.

print(f"KNN best k: {knnGridSearch.best_params_['n_neighbors']}")
print(f"KNN best p: {knnGridSearch.best_params_['p']}")
print(f"KNN cross-validation accuracy: {knnGridSearch.best_score_:.4f}")
print(f"KNN test set accuracy: {knnTestAccuracy:.4f}")
print()

print(f"Decision Tree best max_depth: {decisionTreeGridSearch.best_params_['max_depth']}")
print(f"Decision Tree best min_samples_split: {decisionTreeGridSearch.best_params_['min_samples_split']}")
print(f"Decision Tree best min_samples_leaf: {decisionTreeGridSearch.best_params_['min_samples_leaf']}")
print(f"Decision Tree cross-validation accuracy: {decisionTreeGridSearch.best_score_:.4f}")
print(f"Decision Tree test set accuracy: {decisionTreeTestAccuracy:.4f}")
print()

print(f"AdaBoost best n_estimators: {adaBoostGridSearch.best_params_['n_estimators']}")
print(f"AdaBoost best learning_rate: {adaBoostGridSearch.best_params_['learning_rate']:.4f}")
print(f"AdaBoost cross-validation accuracy: {adaBoostGridSearch.best_score_:.4f}")
print(f"AdaBoost test set accuracy: {adaBoostTestAccuracy:.4f}")
print()

print(f"Gradient Boost best max_depth: {gradientBoostGridSearch.best_params_['max_depth']}")
print(f"Gradient Boost best n_estimators: {gradientBoostGridSearch.best_params_['n_estimators']}")
print(f"Gradient Boost best learning_rate: {gradientBoostGridSearch.best_params_['learning_rate']:.4f}")
print(f"Gradient Boost cross-validation accuracy: {gradientBoostGridSearch.best_score_:.4f}")
print(f"Gradient Boost test set accuracy: {gradientBoostTestAccuracy:.4f}")
print()

print(f"RF best n_estimators: {randomForestGridSearch.best_params_['n_estimators']}")
print(f"RF best max_leaf_nodes: {randomForestGridSearch.best_params_['max_leaf_nodes']}")
print(f"RF cross-validation accuracy: {randomForestGridSearch.best_score_:.4f}")
print(f"RF test set accuracy: {randomForestTestAccuracy:.4f}")
print(f"RF test set macro average F1: {randomForestMacroF1:.4f}")
print(f"RF test set weighted average F1: {randomForestWeightedF1:.4f}")
print()

print(f"SVM best C: {svmGridSearch.best_params_['C']}")
print(f"SVM best gamma: {svmGridSearch.best_params_['gamma']:.4f}")
print(f"SVM best kernel: {svmGridSearch.best_params_['kernel']}")
print(f"SVM cross-validation accuracy: {svmGridSearch.best_score_:.4f}")
print(f"SVM test set accuracy: {svmTestAccuracy:.4f}")

### Test your code

`test-before.csv` contains 6 feature columns (`a1`-`a6`) while `rice-final2.csv`
contains 7 named features. The next cell performs schema diagnostics and only
runs external evaluation when a deterministic, evidence-based mapping can be
established.


In [ ]:
# Load the test dataset to test out your model
from pathlib import Path

import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

testFileName = "test-before.csv"
testPath = Path(testFileName)

if not testPath.exists():
    raise FileNotFoundError(f"External test dataset not found: {testPath.resolve()}")

# Read dataset
testDf = pd.read_csv(testPath, na_values="?")
if testDf.shape[1] < 2:
    raise ValueError("External dataset must contain at least one feature column and one class column.")

# Separate features and labels
xExternalRaw = testDf.iloc[:, :-1].copy()
yExternalRaw = testDf.iloc[:, -1].copy()

# Convert labels after stripping whitespace
labelMap = {"class1": 0, "class2": 1}
yExternalClean = yExternalRaw.astype(str).str.strip()
yExternal = yExternalClean.map(labelMap)

if yExternal.isnull().any():
    unknownLabels = sorted(yExternalClean[yExternal.isnull()].unique())
    raise ValueError(f"Unexpected external class labels found: {unknownLabels}")

yExternal = yExternal.astype(int).to_numpy()

trainingFeatureNames = list(X.columns)
externalFeatureNames = list(xExternalRaw.columns)

print("External test dataset loaded successfully.")
print(f"Dataset shape: {testDf.shape}")
print(f"Feature matrix shape: {xExternalRaw.shape}")
print(f"Target vector shape: {yExternal.shape}")
print(f"Training feature names ({len(trainingFeatureNames)}): {trainingFeatureNames}")
print(f"External feature names ({len(externalFeatureNames)}): {externalFeatureNames}")


def normalize_feature_name(name):
    return str(name).strip().lower()


def get_deterministic_feature_mapping(train_names, external_names):
    train_lookup = {normalize_feature_name(name): name for name in train_names}
    mapping = {}
    unresolved = []

    for external_name in external_names:
        normalized = normalize_feature_name(external_name)
        if normalized in train_lookup:
            mapping[external_name] = train_lookup[normalized]
        else:
            unresolved.append(external_name)

    if unresolved:
        return None, unresolved
    return mapping, []


featureMapping, unresolvedExternalFeatures = get_deterministic_feature_mapping(
    trainingFeatureNames,
    externalFeatureNames
)

if featureMapping is None:
    normalizedTrainNames = {normalize_feature_name(name) for name in trainingFeatureNames}
    normalizedExternalNames = {normalize_feature_name(name) for name in externalFeatureNames}
    overlap = sorted(normalizedTrainNames.intersection(normalizedExternalNames))

    print(f"Normalized feature-name overlap: {overlap if overlap else 'none'}")
    print("No deterministic feature mapping can be proven from repository evidence.")

    trainNumeric = X.apply(pd.to_numeric, errors="coerce")
    externalNumeric = xExternalRaw.apply(pd.to_numeric, errors="coerce")

    print("\nTraining feature diagnostics:")
    for column in trainingFeatureNames:
        columnSeries = trainNumeric[column]
        print(
            f"  {column}: nan={int(columnSeries.isna().sum())}, "
            f"min={columnSeries.min():.4f}, max={columnSeries.max():.4f}"
        )

    print("\nExternal feature diagnostics:")
    for column in externalFeatureNames:
        columnSeries = externalNumeric[column]
        print(
            f"  {column}: nan={int(columnSeries.isna().sum())}, "
            f"min={columnSeries.min():.4f}, max={columnSeries.max():.4f}"
        )

    raise ValueError(
        "Cannot evaluate on test-before.csv because no deterministic 7-to-6 feature mapping is documented. "
        "Training uses 7 named rice features, but external data has 6 unmatched generic features (a1-a6). "
        "A proven feature definition is required before external evaluation."
    )

# Apply the proven mapping to create a shared feature space.
alignedFeatureSpace = [featureMapping[externalName] for externalName in externalFeatureNames]
print(f"Final aligned feature space ({len(alignedFeatureSpace)}): {alignedFeatureSpace}")

xTrainAlignedRaw = X.loc[:, alignedFeatureSpace].copy()
xExternalAlignedRaw = xExternalRaw.copy()
xExternalAlignedRaw.columns = alignedFeatureSpace

xTrainAlignedNumeric = xTrainAlignedRaw.apply(pd.to_numeric, errors="coerce")
xExternalAlignedNumeric = xExternalAlignedRaw.apply(pd.to_numeric, errors="coerce")

fullyMissingTrainColumns = xTrainAlignedNumeric.columns[xTrainAlignedNumeric.isna().all()].tolist()
if fullyMissingTrainColumns:
    raise ValueError(f"Mapped training columns are fully missing: {fullyMissingTrainColumns}")

fullyMissingExternalColumns = xExternalAlignedNumeric.columns[xExternalAlignedNumeric.isna().all()].tolist()
if fullyMissingExternalColumns:
    raise ValueError(f"Mapped external columns are fully missing: {fullyMissingExternalColumns}")

# Fit preprocessing on the aligned training representation only.
alignedImputer = SimpleImputer(strategy="mean")
xTrainAlignedImputed = alignedImputer.fit_transform(xTrainAlignedNumeric)
xExternalAlignedImputed = alignedImputer.transform(xExternalAlignedNumeric)

alignedScaler = MinMaxScaler()
xTrainAlignedScaled = alignedScaler.fit_transform(xTrainAlignedImputed)
xExternalAlignedScaled = alignedScaler.transform(xExternalAlignedImputed)

# Fit final models on the aligned training representation.
allModels = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=0).fit(xTrainAlignedScaled, yProcessed),
    "Naive Bayes": GaussianNB().fit(xTrainAlignedScaled, yProcessed),
    "KNN": KNeighborsClassifier(metric="minkowski", **knnGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "Decision Tree": DecisionTreeClassifier(random_state=0, **decisionTreeGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "AdaBoost": AdaBoostClassifier(random_state=0, **adaBoostGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "Gradient Boosting": GradientBoostingClassifier(random_state=0, **gradientBoostGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "Random Forest": RandomForestClassifier(
        criterion="entropy",
        max_features="sqrt",
        random_state=0,
        **randomForestGridSearch.best_params_
    ).fit(xTrainAlignedScaled, yProcessed),
    "SVM": SVC(random_state=0, **svmGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed)
}

print("\nExternal test results on test-before.csv")
print("-" * 60)

for modelName, model in allModels.items():
    yExternalPred = model.predict(xExternalAlignedScaled)

    externalAccuracy = accuracy_score(yExternal, yExternalPred)
    externalMacroF1 = f1_score(yExternal, yExternalPred, average="macro")
    externalWeightedF1 = f1_score(yExternal, yExternalPred, average="weighted")

    print(f"{modelName} external test accuracy: {externalAccuracy:.4f}")
    print(f"{modelName} external test macro average F1: {externalMacroF1:.4f}")
    print(f"{modelName} external test weighted average F1: {externalWeightedF1:.4f}")
    print()


## **3. Reflection and Discussion**

